# BOJ Momentum Analysis (Final Feature Set)

特徴量設計:
1. 全BOJスプレッド (1-8): 前日値 + 5日MA乖離
2. その他項目 (USDJPY等): 5日MA乖離のみ（水準値は除外）
3. ノイズ除去: MPM翌日から5日間を完全に除外

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. データの読み込み
df = pd.read_excel('data/BOJ_data.xlsx')
df = df.iloc[1:].copy()
df['日付'] = pd.to_datetime(df['日付'], format='%Y年%m月%d日')

swap_cols = [f'JPBOJ{i}ONI=TRDT (MID_PRICE)' for i in range(1, 9)]
tona_col = 'JPY1DOIS=ICAP (MID_PRICE)'
jpy_col = 'JPY= (MID_PRICE)'
market_cols = ['JGBc1 (TRDPRC_1)', '.N225 (TRDPRC_1)', '.DXY (TRDPRC_1)']

all_cols = swap_cols + [tona_col, jpy_col] + market_cols
for col in all_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values('日付').reset_index(drop=True)
df.dropna(subset=[swap_cols[0], tona_col], inplace=True)
print(f'Loaded {len(df)} rows')

In [ ]:
# 2. MPM日程
mpm_dates = pd.to_datetime([
    '2024-01-23', '2024-03-19', '2024-04-26', '2024-06-14', '2024-07-31', '2024-09-20', '2024-10-31', '2024-12-19',
    '2025-01-24', '2025-03-19', '2025-04-30', '2025-06-17', '2025-07-31', '2025-09-19', '2025-10-30', '2025-12-19',
    '2026-01-23', '2026-03-19', '2026-04-28', '2026-06-16', '2026-07-31', '2026-09-18', '2026-10-30', '2026-12-18'
])
def get_next_mpm(d):
    future = mpm_dates[mpm_dates > d]
    return future[0] if len(future) > 0 else None
df['Next_MPM'] = df['日付'].apply(get_next_mpm)
df['DaysToNextMPM'] = (df['Next_MPM'] - df['日付']).dt.days

In [ ]:
# 3. 特徴量生成
features = []
df_feats = df.copy()
window = 5

# A. BOJスワップ (1-8すべて): Spread_lag1 と Spread_diff_MA5 を追加
for i, col in enumerate(swap_cols):
    spread_col = f'BOJ{i+1}_Spread'
    df_feats[spread_col] = df_feats[col] - df_feats[tona_col]
    
    # 1日前スプレッド
    lag1_name = f'{spread_col}_lag1'
    df_feats[lag1_name] = df_feats[spread_col].shift(1)
    features.append(lag1_name)
    
    # 5日MA乖離
    ma = df_feats[spread_col].shift(1).rolling(window=window).mean()
    diff_name = f'{spread_col}_diff_MA{window}'
    df_feats[diff_name] = df_feats[lag1_name] - ma
    features.append(diff_name)

# B. その他項目: 前日値は入れず、5日MA乖離のみ
for col in [jpy_col] + market_cols:
    lag1 = df_feats[col].shift(1)
    ma = df_feats[col].shift(1).rolling(window=window).mean()
    diff_name = f'{col}_diff_MA{window}'
    df_feats[diff_name] = lag1 - ma
    features.append(diff_name)

features.append('DaysToNextMPM')

# ターゲット: BOJ_3
target_col = swap_cols[2]
df_feats['Target'] = df_feats[target_col].shift(-1)

# C. ノイズ除去: MPM後5日間を除外
exclude_dates = []
for mpm in mpm_dates:
    for offset in range(1, 6):
        exclude_dates.append(mpm + pd.Timedelta(days=offset))

df_feats = df_feats[~df_feats['日付'].isin(exclude_dates)]
df_feats.dropna(inplace=True)

print(f'Final samples: {len(df_feats)}')
print(f'Total Features: {len(features)}')
print("\n--- Feature List (All) ---")
for f in features: print(f' - {f}')
display(df_feats[features + ['Target']].head())

In [ ]:
# 4. 学習と評価
X = df_feats[features]
y = df_feats['Target']
tscv = TimeSeriesSplit(n_splits=5)
all_mae, all_mse, all_dir = [], [], []

for tr, te in tscv.split(X):
    X_train, X_test = X.iloc[tr], X.iloc[te]
    y_train, y_test = y.iloc[tr], y.iloc[te]
    model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42, verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='rmse', callbacks=[lgb.early_stopping(stopping_rounds=30)])
    y_pred = model.predict(X_test)
    all_mae.append(mean_absolute_error(y_test, y_pred))
    all_mse.append(mean_squared_error(y_test, y_pred))
    
    # 方向一致率 (今日の絶対値と比較)
    today_val = df_feats.loc[y_test.index, target_col]
    actual_dir = np.sign(y_test - today_val)
    pred_dir = np.sign(y_pred - today_val)
    all_dir.append(np.mean(actual_dir == pred_dir))

print(f'\n--- Prediction Summary ---')
print(f'MSE: {np.mean(all_mse):.7f}')
print(f'MAE: {np.mean(all_mae)*100:.3f} bps')
print(f'Direction Acc: {np.mean(all_dir):.2%}')

In [ ]:
# 5. 可視化
# Importance (Gain)
imp = pd.DataFrame({'f': features, 'i': model.booster_.feature_importance(importance_type='gain')}).sort_values('i', ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x='i', y='f', data=imp)
plt.title('Feature Importance (Gain)')
plt.show()

# BOJ3 Spread vs MA5 Diff
fig, ax1 = plt.subplots(figsize=(15, 5))
ax1.plot(df_feats['日付'], df_feats['BOJ3_Spread_lag1'], color='blue', alpha=0.3)
ax2 = ax1.twinx()
ax2.bar(df_feats['日付'], df_feats['BOJ3_Spread_diff_MA5'], color='red', alpha=0.4)
plt.title('BOJ3 Spread Level (Blue) vs 5d-MA Momentum (Red Bar)')
plt.show()